# Severity Ranking Framework

This notebook is a starter workflow for the complaint severity section of the project.

The goal is to rank vehicle complaints by how likely they are to include a high-risk safety signal. We are not trying to prove that a vehicle has a confirmed defect. We are using the complaint data to find cases that deserve closer review.

Main target:
- `severity_primary_flag`: `True` when a complaint includes deaths, injuries, fire, or crash signals

Optional sensitivity target:
- `severity_broad_flag`: `True` when a complaint includes the primary signals plus severity-adjacent signals such as medical attention or towing required

Why this is a ranking problem:
- severe complaints are uncommon, so plain accuracy can look good even for a weak model
- we care whether the model puts truly severe complaints near the top of a review list

Output written by this notebook:
- `data/outputs/severity_partner_results.csv`

## 1. Setup

Run this cell first. It imports the packages used in the notebook and finds the repo folders automatically. Make sure your venv is setup and choose that as the kernel when prompted.

If this cell fails, make sure you opened the notebook from the repo and completed the README setup steps.

In [32]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Keep local paths explicit so the notebook stays self-contained
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROJECT_ROOT = PROJECT_ROOT.resolve()

In [33]:
# Keep paths and seeds visible in the notebook instead of importing local modules
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'

RANDOM_SEED = 42
sns.set_theme(style='whitegrid')
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Processed data folder:', PROCESSED_DATA_DIR)
print('Outputs folder:', OUTPUTS_DIR)

Project root: c:\Users\davis\Documents\VCCode Repos\NHTSA-ODI-Complaint-Analytics
Processed data folder: C:\Users\davis\Documents\VCCode Repos\NHTSA-ODI-Complaint-Analytics\data\processed
Outputs folder: C:\Users\davis\Documents\VCCode Repos\NHTSA-ODI-Complaint-Analytics\data\outputs


## 2. Load The Severity Table

This notebook starts from the cleaned case-level severity table. That means the raw complaint files and cleaning rules have already been handled by the src pipeline.

In [34]:
# Verify the processed data is available and load it
severity_path = PROCESSED_DATA_DIR / 'odi_severity_cases.parquet'
if not severity_path.exists():
    raise FileNotFoundError(
        f'Missing {severity_path}. Run the README pipeline setup first so data/processed contains the severity table.'
    )

# Load the severity data and convert ldate into a real datatime
df = pd.read_parquet(severity_path)
df['ldate'] = pd.to_datetime(df['ldate'])

# Create a cleaned text column for the complaint description, null values are replaced with empty strings, multiple spaces are reduced to single spaces, and leading/trailing whitespace is removed
df['cdescr_model_text'] = df['cdescr'].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

print('Rows:', len(df))
print('Columns:', len(df.columns))
display(df[['odino', 'ldate', 'mfr_name', 'maketxt', 'modeltxt', 'yeartxt', 'severity_primary_flag', 'severity_broad_flag']].head())

Rows: 375937
Columns: 62


,odino,ldate,mfr_name,maketxt,modeltxt,yeartxt,severity_primary_flag,severity_broad_flag
0,10816121,2020-10-05,"GENERAL MOTORS, LLC",PONTIAC,VIBE,2007,False,False
1,11289434,2020-01-02,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,CR-V,2019,False,False
2,11289436,2020-01-02,HYUNDAI MOTOR AMERICA,HYUNDAI,SONATA,2018,True,True
3,11290552,2020-01-02,"CHRYSLER (FCA US, LLC)",DODGE,DURANGO,2011,True,True
4,11292384,2020-01-01,HONDA (AMERICAN HONDA MOTOR CO.),HONDA,ACCORD,2018,False,False


## 3. Understand The Target

`severity_primary_flag` is uncommon. That is why this notebook reports ranking metrics such as PR-AUC (like AUC score from an ROC Curve, but uses Precision and Recall on the axes instead of True Positive Rate and False Positive Rate) and recall in the highest-risk rows, not just accuracy.

In [35]:
# TODO: Choose the target you want to model first. Use 'severity_primary_flag' for the main target or 'severity_broad_flag' for the broader sensitivity target.
TARGET_COL = 'severity_primary_flag'

if TARGET_COL is None:
    raise ValueError("TODO: Set TARGET_COL to 'severity_primary_flag' or 'severity_broad_flag' before continuing.")

if TARGET_COL not in {'severity_primary_flag', 'severity_broad_flag'}:
    raise ValueError("TARGET_COL must be 'severity_primary_flag' or 'severity_broad_flag'.")

target_summary = pd.DataFrame({
    'target': ['severity_primary_flag', 'severity_broad_flag'],
    'positive_cases': [int(df['severity_primary_flag'].sum()), int(df['severity_broad_flag'].sum())],
    'positive_rate': [float(df['severity_primary_flag'].mean()), float(df['severity_broad_flag'].mean())]
})
display(target_summary)

year_summary = df.assign(year=df['ldate'].dt.year).groupby('year').agg(
    rows=('odino', 'count'),
    primary_rate=('severity_primary_flag', 'mean'),
    broad_rate=('severity_broad_flag', 'mean')
).reset_index()
display(year_summary)

,target,positive_cases,positive_rate
0,severity_primary_flag,22706,0.060398
1,severity_broad_flag,34071,0.090630


,year,rows,primary_rate,broad_rate
0,2020,58465,0.066843,0.085077
1,2021,49044,0.071568,0.096607
2,2022,53221,0.065425,0.101708
3,2023,62727,0.061887,0.094026
4,2024,67432,0.049976,0.084826
5,2025,73985,0.052565,0.086288
6,2026,11063,0.06011,0.085329


## 4. Create Time-Aware Splits

The model should learn from older complaints and be tested on later complaints. Do not randomly shuffle rows across years for the final evaluation because that would make the task easier than real future use.

For this notebook:
- train: complaints received through the end of 2024
- validation: complaints received in 2025
- holdout: complaints received in 2026, kept separate until the final comparison

In [36]:
# Split the severity table by received date so validation and holdout stay future-facing

train_mask = df['ldate'] <= pd.Timestamp('2024-12-31')
valid_mask = (df['ldate'] >= pd.Timestamp('2025-01-01')) & (df['ldate'] <= pd.Timestamp('2025-12-31'))
holdout_mask = df['ldate'] >= pd.Timestamp('2026-01-01')

train_df = df.loc[train_mask].copy()
valid_df = df.loc[valid_mask].copy()
holdout_df = df.loc[holdout_mask].copy()

split_summary = pd.DataFrame({
    'split': ['train_through_2024', 'valid_2025', 'holdout_2026'],
    'rows': [len(train_df), len(valid_df), len(holdout_df)],
    'positive_rate': [train_df[TARGET_COL].mean(), valid_df[TARGET_COL].mean(), holdout_df[TARGET_COL].mean()]
})
display(split_summary)

,split,rows,positive_rate
0,train_through_2024,290889,0.062402
1,valid_2025,73985,0.052565
2,holdout_2026,11063,0.060110


## 5. Metric Helpers

PR-AUC is the main metric here. It asks whether severe cases get higher scores than non-severe cases, which is more useful than accuracy when severe cases are rare.

`recall_top_5pct` answers a practical question: if we reviewed only the top 5% highest-risk complaints, what share of all severe complaints would we catch?
The same logic applies to the different percent scores.

Brier score is a proper scoring function that measures how accurately binary probabilistic events are predicted. A score closer to 0 is better and it's generally used to compare how well calibrated different models are (as opposed to something like F1 score which is better at telling you how well you decisions within a model perform). We'll use it to compare benchmark models against a baseline model.

In [37]:
severity_results = []

# Use this function to get the positive class scores from a model, it will use predict_proba if available and
# fall back to decision_function with a sigmoid transformation if not
def get_positive_scores(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        return proba[:, 1]
    scores = model.decision_function(X)
    return 1 / (1 + np.exp(-scores))

# This function calculates the recall at the top fraction of cases based on the model's scores. It sorts the scores,
# selects the top fraction, and computes how many of the true positives are captured in that top fraction compared to
# the total number of positives.
def recall_at_top_fraction(y_true, score, fraction=0.05):
    y_true = np.asarray(y_true).astype(bool)
    score = np.asarray(score)
    n_top = max(1, int(np.ceil(len(score) * fraction)))
    top_idx = np.argsort(score)[-n_top:]
    total_positive = y_true.sum()
    if total_positive == 0:
        return np.nan
    return y_true[top_idx].sum() / total_positive

def build_linear_sgd_model():
    return SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-5,
        max_iter=100,
        tol=1e-3,
        class_weight='balanced',
        random_state=RANDOM_SEED,
        early_stopping=True,
        validation_fraction=0.05,
        n_iter_no_change=5
    )

# This function evaluates the full set of metric for a give model and dataset then appends the results to the severity_results list
def evaluate_scores(model_name, split_name, y_true, score, threshold=0.5):
    pred = score >= threshold
    row = {
        'model': model_name,
        'split': split_name,
        'rows': len(y_true),
        'positive_rate': float(np.mean(y_true)),
        'pr_auc': float(average_precision_score(y_true, score)),
        'roc_auc': float(roc_auc_score(y_true, score)),
        'f1_at_0_5': float(f1_score(y_true, pred, zero_division=0)),
        'precision_at_0_5': float(precision_score(y_true, pred, zero_division=0)),
        'recall_at_0_5': float(recall_score(y_true, pred, zero_division=0)),
        'recall_top_5pct': float(recall_at_top_fraction(y_true, score, 0.05)),
        'recall_top_10pct': float(recall_at_top_fraction(y_true, score, 0.10)),
        'brier_score': float(brier_score_loss(y_true, score))
    }
    severity_results.append(row)
    return row

def show_results():
    results_df = pd.DataFrame(severity_results)
    display(results_df.sort_values(['split', 'pr_auc'], ascending=[True, False]))
    return results_df

## 6. Naive Baseline

A baseline is a simple model we must beat. If our real models cannot beat it, they are not useful.

This one uses a simple dummy model that predicts the same probability for all cases based on the overall positive rate in the training data. We will use the DummyClassifier from scikit-learn with the 'prior' strategy, which means it will predict the class probabilities based on the distribution of classes in the training data (i.e. for our binary example, if 1% of the target variable is True then DummyClassifier will use predict_proba to match that distribution). After fitting the dummy model, we will get the predicted probabilities for the validation set and evaluate the metrics using our evaluate_scores function.

In [38]:
dummy_model = DummyClassifier(strategy='prior', random_state=RANDOM_SEED)
dummy_model.fit(train_df[['odino']], train_df[TARGET_COL])

dummy_valid_score = dummy_model.predict_proba(valid_df[['odino']])[:, 1]
dummy_scores = evaluate_scores('dummy_prior', 'valid_2025', valid_df[TARGET_COL].to_numpy(), dummy_valid_score)
show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.5,0.0,0.0,0.0,0.05837,0.119054,0.049898


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.5,0.0,0.0,0.0,0.05837,0.119054,0.049898


## 7. Structured Baseline

This model uses the leakage-safe structured core: vehicle identity, complaint context, timing, and missingness fields.

It intentionally excludes `injured`, `deaths`, `fire`, `crash`, `medical_attn`, `vehicles_towed_yn`, and `datea` because those fields define or closely shadow the severity target.

In [39]:
# Derive complaint timing fields up front so the structured core can use them safely
for frame in [train_df, valid_df, holdout_df]:
    frame['complaint_year'] = frame['ldate'].dt.year.astype('int64')
    frame['complaint_month'] = frame['ldate'].dt.month.astype('int64')

# Leakage-safe structured core from the cleaning policy
structured_cat_features = [
    'mfr_name',
    'maketxt',
    'modeltxt',
    'state',
    'cmpl_type',
    'drive_train',
    'fuel_sys',
    'fuel_type',
    'trans_type',
    'police_rpt_yn',
    'repaired_yn',
    'orig_owner_yn'
]

structured_num_features = [
    'yeartxt',
    'miles',
    'veh_speed',
    'lag_days_safe',
    'miles_missing_flag',
    'veh_speed_missing_flag',
    'faildate_trusted_flag',
    'faildate_untrusted_flag',
    'component_count',
    'row_count',
    'complaint_year',
    'complaint_month'
]

structured_cat_features = [c for c in structured_cat_features if c in df.columns]
structured_num_features = [c for c in structured_num_features if c in df.columns]

def prepare_structured_features(source_df):
    X = source_df[structured_cat_features + structured_num_features].copy()
    for col in structured_cat_features:
        X[col] = X[col].astype('string').fillna('missing').astype(str)
    for col in structured_num_features:
        X[col] = pd.to_numeric(X[col], errors='coerce').astype('float64')
    return X

X_train_structured = prepare_structured_features(train_df)
X_valid_structured = prepare_structured_features(valid_df)
X_holdout_structured = prepare_structured_features(holdout_df)

structured_preprocess = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50))
        ]), structured_cat_features),
        ('num', Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale', StandardScaler())
        ]), structured_num_features)
    ],
    remainder='drop'
)

X_train_structured_matrix = structured_preprocess.fit_transform(X_train_structured)
X_valid_structured_matrix = structured_preprocess.transform(X_valid_structured)
X_holdout_structured_matrix = structured_preprocess.transform(X_holdout_structured)

structured_model = build_linear_sgd_model()
structured_model.fit(X_train_structured_matrix, train_df[TARGET_COL])
structured_valid_score = get_positive_scores(structured_model, X_valid_structured_matrix)
evaluate_scores('structured_core_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), structured_valid_score)
show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.46593,0.422217,0.461301,0.085323
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.00000,0.058370,0.119054,0.049898


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.00000,0.058370,0.119054,0.049898
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.46593,0.422217,0.461301,0.085323


## 8. Text Baseline

This model uses the complaint narrative text. TF-IDF turns the text into numbers by giving more weight to words and phrases that are distinctive.

This section now uses two text views:
- word TF-IDF catches words and short phrases such as `air bag`, `caught fire`, or `brake failure`
- character TF-IDF catches smaller spelling patterns and helps when narratives contain typos or inconsistent wording

The vocabularies are capped so the notebook stays runnable on local hardware while still covering the strongest phrases and typo patterns.

In [40]:
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (3, 5)
TEXT_MIN_DF = 5
WORD_MAX_FEATURES = 20000
CHAR_MAX_FEATURES = 10000

train_text = train_df['cdescr_model_text']
valid_text = valid_df['cdescr_model_text']
holdout_text = holdout_df['cdescr_model_text']

text_preprocess = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=WORD_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        max_df=0.995,
        max_features=WORD_MAX_FEATURES,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=CHAR_NGRAM_RANGE,
        min_df=TEXT_MIN_DF,
        max_features=CHAR_MAX_FEATURES,
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        dtype=np.float32
    ))
])

X_train_text_matrix = text_preprocess.fit_transform(train_text)
X_valid_text_matrix = text_preprocess.transform(valid_text)
X_holdout_text_matrix = text_preprocess.transform(holdout_text)

text_model = build_linear_sgd_model()
text_model.fit(X_train_text_matrix, train_df[TARGET_COL])
text_valid_score = get_positive_scores(text_model, X_valid_text_matrix)
evaluate_scores('text_tfidf_word_char_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), text_valid_score)
show_results()

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
2,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.750836,0.874518,0.040762
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.465930,0.422217,0.461301,0.085323
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.058370,0.119054,0.049898


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.058370,0.119054,0.049898
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.465930,0.422217,0.461301,0.085323
2,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.750836,0.874518,0.040762


## 9. Structured + Text Model

This section stacks the leakage-safe structured matrix and the fitted TF-IDF text matrix, then trains one more linear model on the combined sparse table.

Reusing the fitted transforms from sections 7 and 8 keeps the comparison fair and avoids vectorizing the full complaint corpus twice.

In [41]:
X_train_hybrid = sparse.hstack([
    X_train_structured_matrix,
    X_train_text_matrix
], format='csr')
X_valid_hybrid = sparse.hstack([
    X_valid_structured_matrix,
    X_valid_text_matrix
], format='csr')
X_holdout_hybrid = sparse.hstack([
    X_holdout_structured_matrix,
    X_holdout_text_matrix
], format='csr')

hybrid_model = build_linear_sgd_model()
hybrid_model.fit(X_train_hybrid, train_df[TARGET_COL])
hybrid_valid_score = get_positive_scores(hybrid_model, X_valid_hybrid)
evaluate_scores('hybrid_structured_text_sgd', 'valid_2025', valid_df[TARGET_COL].to_numpy(), hybrid_valid_score)

comparison_cols = [
    'model',
    'pr_auc',
    'roc_auc',
    'precision_at_0_5',
    'recall_at_0_5',
    'recall_top_5pct',
    'brier_score'
]
comparison_df = pd.DataFrame(severity_results)
comparison_df = comparison_df.loc[
    comparison_df['split'].eq('valid_2025')
    & comparison_df['model'].isin([
        'structured_core_sgd',
        'text_tfidf_word_char_sgd',
        'hybrid_structured_text_sgd'
    ]),
    comparison_cols
].sort_values('pr_auc', ascending=False)

display(comparison_df)
show_results()

,model,pr_auc,roc_auc,precision_at_0_5,recall_at_0_5,recall_top_5pct,brier_score
2,text_tfidf_word_char_sgd,0.830288,0.963907,0.534929,0.856518,0.750836,0.040762
3,hybrid_structured_text_sgd,0.819780,0.955537,0.679443,0.802263,0.746979,0.025077
1,structured_core_sgd,0.429395,0.716434,0.230417,0.465930,0.422217,0.085323


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
2,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.750836,0.874518,0.040762
3,hybrid_structured_text_sgd,valid_2025,73985,0.052565,0.819780,0.955537,0.735762,0.679443,0.802263,0.746979,0.867318,0.025077
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.465930,0.422217,0.461301,0.085323
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.058370,0.119054,0.049898


,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.058370,0.119054,0.049898
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.465930,0.422217,0.461301,0.085323
2,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.750836,0.874518,0.040762
3,hybrid_structured_text_sgd,valid_2025,73985,0.052565,0.819780,0.955537,0.735762,0.679443,0.802263,0.746979,0.867318,0.025077


## 10. Optional: Final Holdout Check

Only run this after choosing your best model from the 2025 validation results. The holdout is meant to be a final check on 2026 complaints.

In [44]:
# After comparing the validation metrics above, only switch RUN_HOLDOUT to True once you intentionally want the final 2026 check
RUN_HOLDOUT = False
BEST_MODEL_NAME = 'structured_core_sgd'

if RUN_HOLDOUT:
    model_lookup = {
        'dummy_prior': dummy_model,
        'structured_core_sgd': structured_model,
        'text_tfidf_word_char_sgd': text_model,
        'hybrid_structured_text_sgd': hybrid_model
    }
    input_lookup = {
        'dummy_prior': holdout_df[['odino']],
        'structured_core_sgd': X_holdout_structured_matrix,
        'text_tfidf_word_char_sgd': X_holdout_text_matrix,
        'hybrid_structured_text_sgd': X_holdout_hybrid
    }
    if BEST_MODEL_NAME not in model_lookup:
        raise ValueError(f'BEST_MODEL_NAME must be one of {sorted(model_lookup)}')

    holdout_score = get_positive_scores(model_lookup[BEST_MODEL_NAME], input_lookup[BEST_MODEL_NAME])
    evaluate_scores(BEST_MODEL_NAME, 'holdout_2026', holdout_df[TARGET_COL].to_numpy(), holdout_score)
    show_results()
else:
    print('Holdout skipped. Set RUN_HOLDOUT = True only after choosing the best model from validation.')

,model,split,rows,positive_rate,pr_auc,roc_auc,f1_at_0_5,precision_at_0_5,recall_at_0_5,recall_top_5pct,recall_top_10pct,brier_score
5,text_tfidf_word_char_sgd,holdout_2026,11063,0.060110,0.850878,0.966709,0.680950,0.553672,0.884211,0.723308,0.887218,0.044969
4,hybrid_structured_text_sgd,holdout_2026,11063,0.060110,0.837802,0.961883,0.744032,0.665480,0.843609,0.715789,0.873684,0.028683
6,structured_core_sgd,holdout_2026,11063,0.060110,0.485616,0.754508,0.380183,0.296141,0.530827,0.487218,0.524812,0.082749
2,text_tfidf_word_char_sgd,valid_2025,73985,0.052565,0.830288,0.963907,0.658561,0.534929,0.856518,0.750836,0.874518,0.040762
3,hybrid_structured_text_sgd,valid_2025,73985,0.052565,0.819780,0.955537,0.735762,0.679443,0.802263,0.746979,0.867318,0.025077
1,structured_core_sgd,valid_2025,73985,0.052565,0.429395,0.716434,0.308347,0.230417,0.465930,0.422217,0.461301,0.085323
0,dummy_prior,valid_2025,73985,0.052565,0.052565,0.500000,0.000000,0.000000,0.000000,0.058370,0.119054,0.049898


## 11. Save Results

This cell writes a small CSV summary that can be shared with the team.

In [ ]:
results_df = show_results()
results_path = OUTPUTS_DIR / 'severity_partner_results.csv'
results_df.to_csv(results_path, index=False)
print('Wrote:', results_path)

## 12. Next TODOs / Extension Ideas

Use this section as a checklist for improving the severity ranking notebook after the starter baselines run.

Suggested next steps:
- Try `severity_broad_flag` as the target and compare it with `severity_primary_flag`. The broad target will usually catch more cases but may be less specific.
- Tune the hybrid regularization or vocabulary caps and see whether it can beat the text-only model on PR-AUC without giving back the calibration gains.
- Add a simple threshold-tuning cell. For example, pick the score cutoff that gives a useful precision/recall tradeoff on `valid_2025`.
- Add a top-risk review table that shows the highest-scored validation complaints with `odino`, `ldate`, severity flags, model score, and `cdescr_model_text`.
- Add a calibration plot or Brier score check if you want to talk about whether the risk scores behave like probabilities.
- Add one or two charts, such as severity rate by year or score distribution by target value.
- If the model looks useful, run the optional holdout cell once with the best model name and save the final comparison.

Keep the holdout set for the final check only. Use `valid_2025` for experimenting with features, thresholds, and model settings.